Lab 24.3 – Reusable Workflows
Goal: Build a single workflow that:

handles both regression and classification,

wraps preprocessing → model → CV → tuning → save/load,

works for mixed data (numeric + categorical),

returns tidy reports + artifacts.

We’ll demo on:

Classification: Seaborn Titanic (survived)

Regression: California Housing (MedHouseVal)

In [1]:
# pip install scikit-learn pandas numpy seaborn matplotlib joblib
import numpy as np, pandas as pd, seaborn as sns, joblib, json, datetime, sklearn
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Tuple

from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, confusion_matrix,
    mean_squared_error, r2_score
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
np.random.seed(0)

1) A single builder for preprocessing + model

In [2]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    num_cols = X.select_dtypes(include='number').columns
    cat_cols = X.select_dtypes(exclude='number').columns
    numeric = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
    categorical = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                            ('oh',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
    return ColumnTransformer([('num', numeric, num_cols),
                              ('cat', categorical, cat_cols)])

In [3]:
@dataclass
class WorkflowConfig:
    task: str                              # "classification" | "regression"
    model_name: str                        # e.g., "logreg", "rf"
    params: Dict[str, Any]                 # model init params
    grid: Optional[Dict[str, List[Any]]]   # hyperparam grid (pipeline-parameter names)
    cv_folds: int = 5
    test_size: float = 0.2
    random_state: int = 42
    scoring: Optional[str] = None          # default per task if None
    save_path: str = "model_artifact.joblib"
    meta_path: str = "model_meta.json"

In [4]:
def make_estimator(task: str, model_name: str, **params):
    if task == "classification":
        if model_name == "logreg":
            return LogisticRegression(max_iter=2000, **params)
        if model_name == "rf":
            return RandomForestClassifier(n_estimators=400, n_jobs=-1, **params)
        raise ValueError("Unknown classification model")
    elif task == "regression":
        if model_name == "ridge":
            return Ridge(**params)
        if model_name == "rf":
            return RandomForestRegressor(n_estimators=400, n_jobs=-1, **params)
        raise ValueError("Unknown regression model")
    else:
        raise ValueError("task must be 'classification' or 'regression'")

2) The reusable fit → evaluate → tune → save function

In [5]:
def run_workflow(X: pd.DataFrame, y: pd.Series, cfg: WorkflowConfig):
    # Split (stratified if classification)
    strat = y if cfg.task == "classification" else None
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=cfg.test_size, random_state=cfg.random_state, stratify=strat
    )

    pre = build_preprocessor(X)
    est = make_estimator(cfg.task, cfg.model_name, **cfg.params)
    pipe = Pipeline([('prep', pre), ('model', est)])

    # CV (before tuning)
    if cfg.task == "classification":
        scoring = {'acc': 'accuracy', 'roc': 'roc_auc', 'f1':'f1'} if cfg.scoring is None else cfg.scoring
        cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.random_state)
    else:
        scoring = {'rmse':'neg_root_mean_squared_error', 'r2':'r2'} if cfg.scoring is None else cfg.scoring
        cv = KFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.random_state)

    cvres = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    print("=== Baseline CV ===")
    for k,v in cvres.items():
        if k.startswith("test_"):
            print(k.replace("test_",""), ":", np.mean(v).round(3), "±", np.std(v).round(3))

    # Optional grid search
    best = pipe
    best_cv = None
    if cfg.grid:
        gs = GridSearchCV(pipe, cfg.grid, cv=cv,
                          scoring=list(scoring.values())[0] if isinstance(scoring, dict) else scoring,
                          n_jobs=-1).fit(Xtr, ytr)
        best = gs.best_estimator_
        best_cv = gs.best_score_
        print("Best params:", gs.best_params_)
        print("Best CV score:", round(best_cv, 3))

    # Fit on train and evaluate on test
    best.fit(Xtr, ytr)

    if cfg.task == "classification":
        yhat = best.predict(Xte)
        proba = best.predict_proba(Xte)[:,1] if hasattr(best, "predict_proba") else None
        metrics = {
            "accuracy": accuracy_score(yte, yhat),
            "f1": f1_score(yte, yhat),
            "roc_auc": roc_auc_score(yte, proba) if proba is not None else None
        }
    else:
        pred = best.predict(Xte)
        metrics = {
            "rmse": mean_squared_error(yte, pred, squared=False),
            "r2": r2_score(yte, pred)
        }

    print("=== Hold‑out ===")
    for k,v in metrics.items():
        print(k, ":", None if v is None else round(v, 3))

    # Save model + minimal metadata
    joblib.dump(best, cfg.save_path)
    meta = {
        "created_at": datetime.datetime.utcnow().isoformat()+"Z",
        "sklearn": sklearn.__version__,
        "task": cfg.task, "model_name": cfg.model_name,
        "params": cfg.params, "grid": cfg.grid,
        "cv_folds": cfg.cv_folds, "test_size": cfg.test_size,
        "metrics": {k: (None if v is None else float(v)) for k,v in metrics.items()}
    }
    with open(cfg.meta_path, "w") as f: json.dump(meta, f, indent=2)

    return best, metrics, meta

3) Demo A — Classification (Titanic)

In [6]:
df = sns.load_dataset('titanic').drop(columns=['alive'])
target = 'survived'
features = ['pclass','sex','age','sibsp','parch','fare','embarked','class','who','alone']
df = df[features + [target]].dropna(subset=[target]).copy()
df[target] = df[target].astype(int)

X_cls = df[features]
y_cls = df[target]

cfg_cls = WorkflowConfig(
    task="classification",
    model_name="logreg",
    params={"solver":"lbfgs"},
    grid={"model__C":[0.1,1,3,10]},
    scoring=None,  # use default dict
    save_path="titanic_pipe.joblib",
    meta_path="titanic_meta.json"
)

best_cls, metrics_cls, meta_cls = run_workflow(X_cls, y_cls, cfg_cls)

=== Baseline CV ===
acc : 0.822 ± 0.007
roc : 0.866 ± 0.022
f1 : 0.76 ± 0.014
Best params: {'model__C': 10}
Best CV score: 0.817
=== Hold‑out ===
accuracy : 0.832
f1 : 0.773
roc_auc : 0.869


C:\Users\PRASAD\AppData\Local\Temp\ipykernel_15108\2913854971.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.datetime.utcnow().isoformat()+"Z",


Try a tree without changing anything else:

In [7]:
cfg_cls_rf = WorkflowConfig(
    task="classification", model_name="rf", params={"max_depth":None},
    grid={"model__max_depth":[None,4,8,12],"model__min_samples_leaf":[1,2,5]},
    save_path="titanic_rf.joblib", meta_path="titanic_rf_meta.json"
)
_, _, _ = run_workflow(X_cls, y_cls, cfg_cls_rf)

=== Baseline CV ===
acc : 0.808 ± 0.023
roc : 0.872 ± 0.024
f1 : 0.746 ± 0.026
Best params: {'model__max_depth': 12, 'model__min_samples_leaf': 5}
Best CV score: 0.823
=== Hold‑out ===
accuracy : 0.821
f1 : 0.733
roc_auc : 0.849


C:\Users\PRASAD\AppData\Local\Temp\ipykernel_15108\2913854971.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.datetime.utcnow().isoformat()+"Z",


4) Demo B — Regression (California Housing)

In [9]:
import json
import joblib
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, Any, Optional
from datetime import datetime
from sklearn import __version__ as sklearn_version

# Datasets
from sklearn.datasets import fetch_california_housing

# Model Selection & Preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

# Metrics
from sklearn.metrics import (
    mean_squared_error, 
    r2_score,
    accuracy_score, 
    f1_score, 
    roc_auc_score
)

# Try importing root_mean_squared_error (sklearn >= 1.4)
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    # Fallback for older versions
    def root_mean_squared_error(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

@dataclass
class WorkflowConfig:
    task: str  # "regression" or "classification"
    model_name: str
    params: Dict[str, Any]
    grid: Dict[str, Any]
    scoring: Optional[str] = None
    save_path: str = "model.joblib"
    meta_path: str = "meta.json"
    test_size: float = 0.2
    random_state: int = 42
    cv_folds: int = 5

def get_model(name: str, task: str, params: Dict[str, Any]):
    """Factory to get model instance based on name and task."""
    if task == "regression":
        if name == "ridge": return Ridge(**params)
        if name == "rf": return RandomForestRegressor(**params)
    elif task == "classification":
        if name == "logreg": return LogisticRegression(**params)
        if name == "rf": return RandomForestClassifier(**params)
    raise ValueError(f"Unknown model {name} for task {task}")

def run_workflow(X, y, cfg: WorkflowConfig):
    """
    Main workflow: Split -> Preprocess -> GridSearch -> Evaluate -> Save
    """
    print(f"Starting workflow for {cfg.task} with model {cfg.model_name}...")
    
    # 1. Split Data
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=cfg.test_size, random_state=cfg.random_state
    )

    # 2. Preprocessing Pipeline
    # Identify column types automatically
    num_cols = X.select_dtypes(include=[np.number]).columns
    cat_cols = X.select_dtypes(exclude=[np.number]).columns

    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    # sparse_output=False is for sklearn >= 1.2, use sparse=False for older
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])

    # 3. Full Pipeline
    base_model = get_model(cfg.model_name, cfg.task, cfg.params)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', base_model)
    ])

    # 4. Grid Search
    search = GridSearchCV(
        pipeline, 
        cfg.grid, 
        cv=cfg.cv_folds, 
        scoring=cfg.scoring,
        n_jobs=-1,
        verbose=1
    )
    search.fit(Xtr, ytr)
    best_estimator = search.best_estimator_
    print(f"Best params: {search.best_params_}")

    # 5. Evaluation
    if cfg.task == "classification":
        pred = best_estimator.predict(Xte)
        # Check if probability is available
        if hasattr(best_estimator, "predict_proba"):
            prob = best_estimator.predict_proba(Xte)[:, 1]
        else:
            prob = None
            
        metrics = {
            "accuracy": accuracy_score(yte, pred),
            "f1": f1_score(yte, pred, average='weighted'),
            "roc_auc": roc_auc_score(yte, prob) if prob is not None and len(np.unique(yte))==2 else None
        }
    else:
        # Regression metrics
        pred = best_estimator.predict(Xte)
        metrics = {
            "rmse": root_mean_squared_error(yte, pred), # Fixed: using dedicated function
            "r2": r2_score(yte, pred)
        }

    print("=== Hold-out Metrics ===")
    for k, v in metrics.items():
        print(f"{k}: {v}")

    # 6. Save Artifacts
    joblib.dump(best_estimator, cfg.save_path)
    
    meta_data = {
        "created_at": datetime.now().isoformat(),
        "sklearn": sklearn_version,
        "task": cfg.task,
        "model_name": cfg.model_name,
        "params": cfg.params,
        "grid": cfg.grid,
        "cv_folds": cfg.cv_folds,
        "test_size": cfg.test_size,
        "metrics": metrics
    }
    
    with open(cfg.meta_path, "w") as f:
        json.dump(meta_data, f, indent=2)
    
    print(f"Saved model to {cfg.save_path} and meta to {cfg.meta_path}")
    return best_estimator, metrics, meta_data

# --- Execution Example ---
if __name__ == "__main__":
    # 1. Load Data
    cal = fetch_california_housing(as_frame=True)
    X_reg = cal.frame.drop(columns=['MedHouseVal'])
    y_reg = cal.frame['MedHouseVal']

    # 2. Configure
    cfg_reg = WorkflowConfig(
        task="regression",
        model_name="ridge",
        params={"alpha": 3.0, "random_state": 0},
        grid={"model__alpha": [0.3, 1, 3, 10, 30]},
        scoring=None, # Default scoring for reg is R2, can use 'neg_root_mean_squared_error'
        save_path="california_ridge.joblib",
        meta_path="california_ridge_meta.json"
    )

    # 3. Run
    best_reg, metrics_reg, meta_reg = run_workflow(X_reg, y_reg, cfg_reg)


Starting workflow for regression with model ridge...
Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params: {'model__alpha': 0.3}
=== Hold-out Metrics ===
rmse: 0.7455739745013783
r2: 0.5757961364092612
Saved model to california_ridge.joblib and meta to california_ridge_meta.json
